In [1]:
import numpy as np
import torch
import torch.nn as nn


def shuffle_dataset(dataset, seed):
    np.random.seed(seed)
    np.random.shuffle(dataset)
    return dataset


def split_dataset(dataset, ratio):
    n = int(ratio * len(dataset))
    dataset_1, dataset_2 = dataset[:n], dataset[n:]
    return dataset_1, dataset_2

    
torch.set_default_tensor_type('torch.cuda.FloatTensor')
DATASET = "classAByProteinWithHelix_test"
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    print('The code uses GPU...')
else:
    device = torch.device('cpu')
    print('The code uses CPU!!!')

def load_tensor2(file_name, dtype):
    proteins = np.load(file_name + '.npy', allow_pickle=True)
    torch_tensor_protein = []
    for protein in proteins:
        this_protein = []
        for helix in protein:
            this_protein.append(dtype(helix).to(device))
        torch_tensor_protein.append(this_protein)
    return torch_tensor_protein
    
def load_tensor(file_name, dtype):
    return [
        dtype(d).to(device)
        for d in np.load(file_name + '.npy', allow_pickle=True)
    ]

def pack(atoms, adjs, proteins, labels, device):
    atoms_len = 0
    proteins_len = 0
    N = len(atoms)
    atom_num = []
    for atom in atoms:
        atom_num.append(atom.shape[0])
        if atom.shape[0] >= atoms_len:
            atoms_len = atom.shape[0]

    protein_num = []
    for p in proteins:
        for protein in p:
            protein_num.append(protein.shape[0])
            if protein.shape[0] >= proteins_len:
                proteins_len = protein.shape[0]
    print(proteins_len)
    atoms_new = torch.zeros((N, atoms_len, 34), device=device)
    i = 0
    for atom in atoms:
        a_len = atom.shape[0]
        atoms_new[i, :a_len, :] = atom
        i += 1
    adjs_new = torch.zeros((N, atoms_len, atoms_len), device=device)
    i = 0
    for adj in adjs:
        a_len = adj.shape[0]
        adj = adj + torch.eye(a_len, device=device)
        adjs_new[i, :a_len, :a_len] = adj
        i += 1
    #############################
    proteins_news_7 = [
        torch.zeros((N, proteins_len, 100), device=device) for i in range(7)
    ]
    i = 0
    for p in proteins:
        for index, protein in enumerate(p):
            a_len = protein.shape[0]
            proteins_news_7[index][i, :a_len, :] = protein
        i += 1
    ###############################
    labels_new = torch.zeros(N, dtype=torch.long, device=device)

    i = 0
    for label in labels:
        labels_new[i] = label
        i += 1
    return (atoms_new, adjs_new, proteins_news_7, labels_new, atom_num,
            protein_num)


The code uses GPU...


In [2]:
dir_input = ('dataset/' + DATASET + '/word2vec_30/')
compounds = load_tensor(dir_input + 'compounds', torch.FloatTensor)
adjacencies = load_tensor(dir_input + 'adjacencies', torch.FloatTensor)
proteins = load_tensor2(dir_input + 'proteins', torch.FloatTensor)
interactions = load_tensor(dir_input + 'interactions', torch.LongTensor)
"""Create a dataset and split it into train/dev/test."""
dataset = list(zip(compounds, adjacencies, proteins, interactions))
dataset = shuffle_dataset(dataset, 1234)
dataset_train, dataset_dev = split_dataset(dataset, 0.8)

np.random.shuffle(dataset)
N = len(dataset)
i = 0
adjs, atoms, proteins, labels = [], [], [], []
for data in dataset:
    i = i + 1
    atom, adj, protein, label = data
    adjs.append(adj)
    atoms.append(atom)
    proteins.append(protein)
    labels.append(label)
    if i % 8 == 0 or i == N:
        data_pack = pack(atoms, adjs, proteins, labels, device)
        break

KeyboardInterrupt: 

In [3]:
proteins = data_pack[2]
p1 = proteins[0]
p2= proteins[1]
p3= proteins[2]
p4= proteins[3]
p5= proteins[4]
p6= proteins[5]
p7= proteins[6]

In [4]:
input_dim = 100
hid_dim = 64

fc1 = nn.Linear(input_dim, hid_dim)
fc2 = nn.Linear(input_dim, hid_dim)
fc3 = nn.Linear(input_dim, hid_dim)
fc4 = nn.Linear(input_dim, hid_dim)
fc5 = nn.Linear(input_dim, hid_dim)
fc6 = nn.Linear(input_dim, hid_dim)
fc7 = nn.Linear(input_dim, hid_dim)

conv_input1 = fc1(proteins[0])
conv_input1 = conv_input1.permute(0, 2, 1)
conv_input2 = fc2(proteins[1])
conv_input2 = conv_input2.permute(0, 2, 1)
conv_input3 = fc3(proteins[2])
conv_input3 = conv_input3.permute(0, 2, 1)
conv_input4 = fc4(proteins[3])
conv_input4 = conv_input4.permute(0, 2, 1)
conv_input5 = fc5(proteins[4])
conv_input5 = conv_input5.permute(0, 2, 1)
conv_input6 = fc6(proteins[5])
conv_input6 = conv_input6.permute(0, 2, 1)
conv_input7 = fc7(proteins[6])
conv_input7 = conv_input7.permute(0, 2, 1)

In [25]:
conved_result_4dim = [torch.unsqueeze(p, 3) for p in proteins]
concated = torch.cat(conved_result_4dim, dim=3)
position_fuuly = nn.Linear(7, 1)
encoded = position_fuuly(concated)
encoded = torch.squeeze(encoded, 3)
r = nn.ReLU()
encoded = r(encoded)
d = nn.Dropout(0.001)
encoded = d(encoded)


In [26]:
encoded.shape

torch.Size([8, 28, 100])